<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_17_practicum_hashing/note_lesson_17_hashing_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 17 — Практикум П3. Хеш-структури: третя зміна диспетчера таксі

У П1 диспетчерська навчилася рахувати ціну рішення, у П2 — обирати стратегію за властивостями даних. Сьогодні журнал поїздок приходить **у порядку дзвінків, невідсортованим**. Замість чекати зручних даних змінимо **представлення**: один раз перекладемо дані у `dict` / `set` / `Counter` — і далі кожна відповідь займе один крок. Наприкінці зазирнемо всередину хеш-таблиці, порахуємо, скільки вона важить у пам'яті, і побачимо ті самі хеш-таблиці всередині бази даних DuckDB.

Виконуй клітинки **зверху вниз**. Перед клітинками з позначкою **Прогноз** спершу скажи, що буде. Теорія — у книзі: [Урок 17. Практикум П3. Хеш-структури](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_16/).

## 🔁 Пригадай (без підглядання)

1. На яку властивість списку спиралися два вказівники в П2?
2. Яка складність `x in список` і `x in множина`?
3. Що робить `counts[name] = counts.get(name, 0) + 1`?

<details>
<summary>Відповіді</summary>

1. На відсортованість.
2. `O(n)` і `O(1)` в середньому.
3. Збільшує лічильник `name` на 1; для нового ключа `get` дає 0.

</details>

## 0. Журнал зміни

In [ ]:
from typing import NamedTuple


class Trip(NamedTuple):
    id: int
    client: str
    start: str
    end: str
    fare: int


trips = [
    Trip(1120, "Марта", "Поділ", "Оболонь", 230),
    Trip(425, "Олег", "Вокзал", "Поділ", 150),
    Trip(930, "Марта", "Оболонь", "Поділ", 270),
    Trip(612, "Ірина", "Печерськ", "Вокзал", 180),
    Trip(845, "Тарас", "Поділ", "Оболонь", 320),
    Trip(1035, "Олег", "Поділ", "Вокзал", 120),
    Trip(700, "Олена", "Оболонь", "Поділ", 410),
    Trip(510, "Богдан", "Вокзал", "Печерськ", 150),
]

fares = [trip.fare for trip in trips]
print(fares)

## 1. Ваучер на 500 грн: два вказівники з П2

**Прогноз:** що поверне `pair_with_sum` на невідсортованому списку? (`230 + 270 = 500`)

In [ ]:
def pair_with_sum(items, target):
    left, right = 0, len(items) - 1
    steps = 0
    while left < right:
        steps += 1
        total = items[left] + items[right]
        if total == target:
            return (items[left], items[right]), steps
        if total < target:
            left += 1
        else:
            right -= 1
    return None, steps


print(pair_with_sum(fares, 500))

<details>
<summary>Відповідь</summary>

`(None, 7)` — «пари немає». Жодної помилки: алгоритм мовчки спирається на порядок, якого немає.

</details>

In [ ]:
def pair_with_sum_brute(items, target):
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1
            if items[i] + items[j] == target:
                return (i, j), steps
    return None, steps


print(pair_with_sum_brute(fares, 500))

## 🛠 Вправа 1. Two Sum за один прохід

Напиши `pair_with_sum_hash(items, target)`: словник `seen` «сума → позиція»; для кожної суми питаємо, чи бачили **доповнення** `target - value`. Повертає `((позиція1, позиція2), кроки)` або `(None, кроки)`.

In [ ]:
def pair_with_sum_hash(items, target):
    seen = {}
    steps = 0
    # YOUR CODE HERE
    pass
    return None, steps


print(pair_with_sum_hash(fares, 500))
assert pair_with_sum_hash(fares, 500) == ((0, 2), 3)
assert pair_with_sum_hash(fares, 10_000) == (None, 8)
assert pair_with_sum_hash([], 500) == (None, 0)
print("✅ Вправа 1 пройдена")

### Дослід подвоєння

**Прогноз:** у скільки разів зростуть кроки перебору і словника, коли журнал подвоюється?

In [ ]:
for n in [1000, 2000, 4000]:
    values = list(range(1, n + 1))
    _, brute = pair_with_sum_brute(values, 10**9)
    _, fast = pair_with_sum_hash(values, 10**9)
    print(n, brute, fast)

<details>
<summary>Відповідь</summary>

Перебір — ×4 (`O(n²)`), словник — ×2 (`O(n)`).

</details>

## 2. Поїздка за номером

Лінійний пошук на кожен дзвінок проти словника-індексу, побудованого один раз.

In [ ]:
def find_trip(trips, trip_id):
    steps = 0
    for trip in trips:
        steps += 1
        if trip.id == trip_id:
            return trip, steps
    return None, steps


print(find_trip(trips, 845))

## 🛠 Вправа 2. Словник-індекс

Побудуй `by_id` — словник «номер поїздки → поїздка» одним dict comprehension.

In [ ]:
# YOUR CODE HERE

print(by_id[845].client)
assert by_id[845].client == "Тарас"
assert by_id.get(999) is None
assert len(by_id) == len(trips)
print("✅ Вправа 2 пройдена")

**Прогноз:** скільки кроків на 100 дзвінків у журналі з 1000 поїздок — лінійно й через індекс?

In [ ]:
import random

ids = list(range(1000))
random.Random(7).shuffle(ids)
big_journal = [Trip(i, "клієнт", "А", "Б", 100) for i in ids]
calls = list(range(0, 1000, 10))

linear_steps = sum(find_trip(big_journal, trip_id)[1] for trip_id in calls)
index = {trip.id: trip for trip in big_journal}
index_steps = len(big_journal) + len(calls)

print(linear_steps, index_steps)

<details>
<summary>Відповідь</summary>

Лінійно — понад 53 000 (у середньому пів журналу на дзвінок), з індексом — 1000 на побудову + 100 запитів. Для **одного** дзвінка індекс не окупився б.

</details>

## 3. Як словник знаходить ключ: кошики

Кошик = `hash(ключ) % кількість_кошиків`. Для малих цілих `hash(n) == n`.

In [ ]:
buckets = [[] for _ in range(8)]
for trip_id in [1120, 425, 930, 612, 845]:
    buckets[hash(trip_id) % 8].append(trip_id)

for number, bucket in enumerate(buckets):
    print(number, bucket)

**Прогноз:** у який кошик потрапить 1128 і скільки порівнянь знадобиться, щоб його знайти?

In [ ]:
def find_in_table(buckets, key):
    bucket = buckets[hash(key) % len(buckets)]
    comparisons = 0
    for item in bucket:
        comparisons += 1
        if item == key:
            return True, comparisons
    return False, comparisons


buckets[hash(1128) % 8].append(1128)
print(buckets[0])
print(find_in_table(buckets, 1128))
print(find_in_table(buckets, 845))

<details>
<summary>Відповідь</summary>

`1128 % 8 = 0` — кошик 0, де вже є 1120: **колізія**, 2 порівняння. Справжній `dict` тримає кошиків більше, ніж ключів, тож колізій мало і пошук в середньому `O(1)`.

</details>

## 4. Що може бути ключем

Лише **hashable** — незмінні значення. Розкоментуй другий рядок: `TypeError: unhashable type: 'list'`. Потім закоментуй назад.

In [ ]:
route_counts = {("Поділ", "Оболонь"): 2}          # кортеж — можна
# route_counts[["Поділ", "Оболонь"]] = 2          # список — не можна
print(route_counts[("Поділ", "Оболонь")])

## 5. Найпопулярніший маршрут

**Прогноз:** які маршрути та скільки разів порахує наївний лічильник?

In [ ]:
def route_counts_naive(trips):
    counts = {}
    for trip in trips:
        route = (trip.start, trip.end)
        counts[route] = counts.get(route, 0) + 1
    return counts


print(route_counts_naive(trips))

<details>
<summary>Відповідь</summary>

«Поділ → Оболонь» і «Оболонь → Поділ» — окремими ключами по 2. Для диспетчера це один маршрут: потрібен канонічний ключ.

</details>

## 🛠 Вправа 3. Канонічний ключ маршруту

`route_key(trip)` — відсортований кортеж з двох точок, щоб напрямок не мав значення.

In [ ]:
from collections import Counter


def route_key(trip):
    # YOUR CODE HERE
    pass


routes = Counter(route_key(trip) for trip in trips)
print(routes.most_common(2))
assert route_key(trips[0]) == route_key(trips[2]) == ("Оболонь", "Поділ")
assert routes.most_common(1) == [(("Оболонь", "Поділ"), 4)]
print("✅ Вправа 3 пройдена")

## 6. Пастка: словник змінюється під час обходу

Розкоментуй цикл — `RuntimeError: dictionary changed size during iteration`. Правильно — будувати новий словник.

In [ ]:
clients = Counter(trip.client for trip in trips)
# for name in clients:
#     if clients[name] < 2:
#         del clients[name]
regular = {name: count for name, count in clients.items() if count >= 2}
print(regular)

## 7. Розібраний приклад: звіт диспетчера за зміну

In [ ]:
def shift_report(trips, voucher):
    pair, _ = pair_with_sum_hash([trip.fare for trip in trips], voucher)
    voucher_trips = [trips[i].id for i in pair] if pair else None
    routes = Counter(route_key(trip) for trip in trips)
    top_route, top_count = routes.most_common(1)[0]
    clients = Counter(trip.client for trip in trips)
    return {
        "voucher_trips": voucher_trips,
        "top_route": " — ".join(top_route),
        "top_route_trips": top_count,
        "regular_clients": [name for name, count in clients.items() if count >= 2],
    }


report = shift_report(trips, 500)
for key, value in report.items():
    print(key, value)

## 🛠 Вправа 4. Перший разовий клієнт

`first_unique(names)` — перше за журналом ім'я, яке трапляється рівно один раз, або `None`. Два проходи: `Counter`, потім пошук по **списку**. Без `names.count(...)` у циклі.

In [ ]:
def first_unique(names):
    # YOUR CODE HERE
    pass


print(first_unique([trip.client for trip in trips]))
assert first_unique([trip.client for trip in trips]) == "Ірина"
assert first_unique(["Марта", "Марта"]) is None
assert first_unique([]) is None
print("✅ Вправа 4 пройдена")

## 🛠 Вправа 5. Анаграми — класика співбесід

- `is_anagram(a, b)` — `Counter` літер, без урахування регістру;
- `group_anagrams(words)` — групи в порядку першої появи, ключ — `"".join(sorted(word.lower()))`, один прохід.

In [ ]:
def is_anagram(a, b):
    # YOUR CODE HERE
    pass


def group_anagrams(words):
    # YOUR CODE HERE
    pass


assert is_anagram("Літо", "тіло") and not is_anagram("літо", "літа")
words = ["літо", "клоун", "тіло", "таксі", "уклон", "кит"]
print(group_anagrams(words))
assert group_anagrams(words) == [["літо", "тіло"], ["клоун", "уклон"], ["таксі"], ["кит"]]
assert group_anagrams([]) == []
print("✅ Вправа 5 пройдена")

## 8. Хеш зсередини

У розділі 3 кошик рахувався як `hash(ключ) % кількість_кошиків`. Подивимося, що повертає `hash()` для різних ключів.

In [ ]:
for key in [237, 1120, "JFK Airport", ("Поділ", "Оболонь"), 12.5]:
    print(f"{key!r:<24} hash = {hash(key):>21}   кошик з 8: {hash(key) % 8}")

- `hash(ціле) == ціле` — тому в розділі 3 було видно, куди піде кожен номер.
- hash рядка — велике «випадкове» число. Перезапусти середовище (Runtime → Restart) і запусти клітинку ще раз: hash рядка **зміниться**, hash числа — ні. Python навмисно перемішує хеші рядків при кожному запуску (`PYTHONHASHSEED`), щоб зловмисник не міг підібрати ключі, які потраплять в один кошик. Тому `set` рядків друкується щоразу в іншому порядку.
- Всередині одного запуску hash ключа незмінний — інакше словник не знайшов би свій ключ.

**Прогноз:** скільки ключів буде в словнику `{1: "int", 1.0: "float", True: "bool"}`?

In [ ]:
prices = {1: "int", 1.0: "float", True: "bool"}
print(prices, len(prices))
print(1 == 1.0 == True, hash(1) == hash(1.0) == hash(True))

<details>
<summary>Відповідь</summary>

Один ключ: `{1: 'bool'}`. Словник шукає ключ так: спершу кошик за `hash`, потім перевірка `==`. Для `1`, `1.0` і `True` збігається і те, і те — для словника це **той самий ключ**. Значення перезаписується, а ключ лишається першим.

</details>

### Таблиця росте сама

Справжній `set` / `dict` тримає щонайменше третину комірок порожніми. Коли заповнення перевищує 2/3, Python виділяє більшу таблицю і **перекладає в неї всі ключі** — бо `hash % кількість_комірок` тепер інший. Побачимо це за розміром множини:

In [ ]:
import sys

seen = set()
previous = sys.getsizeof(seen)
print(f"порожня множина: {previous} байт")
for n in range(1, 100):
    seen.add(n)
    size = sys.getsizeof(seen)
    if size != previous:
        print(f"на {n:>2}-му елементі: {previous:>5} → {size:>5} байт — таблиця розширилась")
        previous = size

Перекладання — це `O(n)`, але таблиця щоразу зростає в кілька разів, тож переїзди стають дедалі рідшими. У середньому додавання лишається `O(1)` — як `append` у списку.

Інтерактивну таблицю з комірками, колізіями, розширенням і поганими хеш-функціями дивись у лабораторії `taxi_lab` поруч із ноутбуком (`streamlit run app.py`).

## 9. Скільки це важить

Швидкість словника і множини не безкоштовна.

**Прогноз:** 100 000 номерів поїздок. У скільки разів `set` займе більше пам'яті, ніж `list`?

In [ ]:
ids = list(range(10_000_000, 10_100_000))       # 100 000 номерів поїздок

for name, container in [("list", ids), ("tuple", tuple(ids)), ("set", set(ids)), ("dict", dict.fromkeys(ids))]:
    size = sys.getsizeof(container)
    print(f"{name:<6} {size / 1024:>8,.0f} КБ   {size / len(ids):5.1f} байт на елемент")

print("одне число:", sys.getsizeof(ids[0]), "байт")

<details>
<summary>Відповідь</summary>

Приблизно в 5 разів. `list` і `tuple` зберігають лише **посилання** на числа — 8 байт на елемент. `set` зберігає для кожного елемента ще й його `hash`, тримає третину комірок порожніми, а кількість комірок — степінь двійки. `dict` — ще більше: ключ, значення і hash.

</details>

`sys.getsizeof` рахує лише **сам контейнер**. Числа — окремі об'єкти по 28 байт (int у Python — повноцінний об'єкт, не 8 байт як у C). Рядки теж мають ціну:

In [ ]:
for text in ["Midtown", "Мідтаун", "Мідтаун🚕"]:
    print(f"{text!r:<12} {len(text)} символів → {sys.getsizeof(text)} байт")

Латиниця — 1 байт на символ, кирилиця — 2, а один емодзі робить 4 байти для **кожного** символу рядка.

## 🛠 Вправа 6. Глибокий розмір

Напиши `deep_size(obj, seen=None)` — розмір об'єкта разом з усім, на що він посилається:

- `sys.getsizeof(obj)` плюс `deep_size` кожного елемента (для `dict` — і ключів, і значень; для `list`, `tuple`, `set`, `frozenset` — елементів);
- один і той самий об'єкт може бути в контейнері кілька разів — рахуй його **один раз**. Для цього тримай множину `seen` з `id()` уже порахованих об'єктів: перевірка «вже рахували?» — `O(1)`.

`Trip` — це `NamedTuple`, тобто кортеж, тож його поля теж порахуються.

In [ ]:
def deep_size(obj, seen=None):
    """Розмір obj разом з усім, на що він посилається; кожен об'єкт — один раз."""
    if seen is None:
        seen = set()
    # YOUR CODE HERE
    pass


big = 10**9 + 7
shared = [big, big, big]                 # одне число, три посилання на нього
print("список з трьох посилань:", deep_size(shared), "байт")
print("одна поїздка:", deep_size(trips[0]), "байт")
print("журнал з", len(trips), "поїздок:", deep_size(trips), "байт, сам список —", sys.getsizeof(trips))

assert deep_size([]) == sys.getsizeof([])
assert deep_size(shared) == sys.getsizeof(shared) + sys.getsizeof(big)
assert deep_size({"a": [1, 2]}) == (sys.getsizeof({"a": [1, 2]}) + sys.getsizeof("a")
                                    + sys.getsizeof([1, 2]) + sys.getsizeof(1) + sys.getsizeof(2))
assert deep_size(trips) > 10 * sys.getsizeof(trips)
print("✅ Вправа 6 пройдена")

Самий список `trips` важить кілька сотень байт, а з поїздками, рядками й числами — у десятки разів більше. Тому великі дані — мільйони поїздок — не тримають у Python-списках. Їх тримають у **базі даних**, а в Python забирають уже відповідь.

## 10. Затравка: реляційна база даних

Реляційна база зберігає дані **таблицями**: рядок — як кортеж, стовпець має тип. Таблиці пов'язуються **ключами**. А запити пишуть мовою **SQL**: описуємо, *що* хочемо отримати, а база сама вирішує, *як*.

Візьмемо [DuckDB](https://duckdb.org/) — базу, що працює просто всередині Python, без сервера. Якщо її немає: `!pip install duckdb`.

In [ ]:
import duckdb

con = duckdb.connect()                          # база в пам'яті
con.execute("""
    CREATE TABLE trips (
        id INTEGER PRIMARY KEY,
        client VARCHAR,
        start VARCHAR,
        finish VARCHAR,
        fare INTEGER
    )
""")
con.executemany("INSERT INTO trips VALUES (?, ?, ?, ?, ?)", trips)   # Trip — кортеж, підходить як є
print(con.sql("SELECT * FROM trips ORDER BY id"))

### `GROUP BY` — це `Counter`

**Прогноз:** що поверне запит нижче? Порівняй з `Counter` у наступному рядку.

In [ ]:
print(con.sql("""
    SELECT client, count(*) AS trips, sum(fare) AS total
    FROM trips
    GROUP BY client
    ORDER BY trips DESC, client
"""))
print(Counter(trip.client for trip in trips).most_common())

### `PRIMARY KEY` — це ключ словника

`id` оголошено первинним ключем: як ключ `dict`, він унікальний. Але реагують на повтор по-різному.

In [ ]:
index = {trip.id: trip for trip in trips}
index[1120] = Trip(1120, "Хтось", "А", "Б", 100)          # dict мовчки перезаписує
print("dict:", len(index), "поїздок, 1120 тепер —", index[1120].client)

try:
    con.execute("INSERT INTO trips VALUES (1120, 'Хтось', 'А', 'Б', 100)")
except duckdb.ConstraintException as error:
    print("база не прийняла повтор ключа:", str(error).splitlines()[0])

### `JOIN` — це словник-індекс

Довідник районів — окрема таблиця. Щоб для кожної поїздки дізнатися тип району посадки, таблиці **з'єднують** за ключем. Команда `EXPLAIN` показує, як база виконає запит.

**Прогноз:** які слова з цього уроку ти побачиш у плані?

In [ ]:
con.execute("CREATE TABLE districts (name VARCHAR PRIMARY KEY, kind VARCHAR)")
con.executemany("INSERT INTO districts VALUES (?, ?)",
                [("Поділ", "центр"), ("Печерськ", "центр"), ("Вокзал", "центр"), ("Оболонь", "спальний")])

sql = """
    SELECT d.kind, count(*) AS trips, round(avg(t.fare), 1) AS avg_fare
    FROM trips AS t
    JOIN districts AS d ON t.start = d.name
    GROUP BY d.kind
    ORDER BY d.kind
"""
print(con.sql(sql))

kinds = {"Поділ": "центр", "Печерськ": "центр", "Вокзал": "центр", "Оболонь": "спальний"}   # те саме на Python
print(Counter(kinds[trip.start] for trip in trips))

plan = con.execute("EXPLAIN " + sql).fetchall()[0][1]
print(plan)
assert "HASH_JOIN" in plan and "HASH_GROUP_BY" in plan

<details>
<summary>Відповідь</summary>

`HASH_JOIN` і `HASH_GROUP_BY`. Для `JOIN` база кладе довідник у **хеш-таблицю** за ключем `name` і для кожної поїздки робить пошук за `O(1)` — як `kinds[trip.start]`. Для `GROUP BY` — хеш-таблиця «ключ групи → лічильник», як `Counter`. Бази даних побудовані на тих самих ідеях, що й цей урок.

</details>

### Необов'язково: 3 мільйони справжніх поїздок

DuckDB уміє читати файл parquet просто з мережі. Це журнал жовтого таксі Нью-Йорка за січень 2023 року з офіційного сайту NYC Taxi & Limousine Commission: понад 3 мільйони поїздок. Потрібен інтернет.

In [ ]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
try:
    print(con.sql(f"""
        SELECT PULocationID AS pickup_zone, count(*) AS trips, round(avg(fare_amount), 2) AS avg_fare
        FROM read_parquet('{url}')
        GROUP BY pickup_zone
        ORDER BY trips DESC
        LIMIT 5
    """))
except Exception as error:                      # немає мережі чи розширення httpfs
    print("Не вдалося прочитати дані з мережі:", type(error).__name__)

Мільйони рядків за частку секунди: база читає з файлу лише потрібні стовпці, рахує на C і не створює мільйони Python-об'єктів по 28 байт кожен. Далі в курсі — модуль «Бази даних»: таблиці, ключі, SQL і PostgreSQL. Більше прикладів зі справжніми поїздками, назвами зон і картою — у лабораторії `taxi_lab`.

## ✅ Самоперевірка

1. Чому два вказівники мовчки помиляються на невідсортованих даних?
2. Що зберігає `seen` у Two Sum і чому словник, а не список?
3. Коли словник-індекс не окупається?
4. Чому список не може бути ключем, а кортеж — може?
5. Навіщо канонічний ключ маршруту?
6. Чому `{1: 'a', 1.0: 'b'}` містить один ключ?
7. Чому `set` зі 100 000 чисел займає в кілька разів більше пам'яті, ніж `list`?
8. Що в базі даних відповідає ключу словника, `Counter` і словнику-індексу?

<details>
<summary>Відповіді</summary>

1. Алгоритм не перевіряє порядок — лише покладається на нього.
2. Побачені суми та позиції; перевірка «чи бачили» в словнику — один крок, у списку — прохід.
3. Коли запит один або їх мало: побудова — n кроків, лінійний пошук — у середньому n / 2.
4. Список змінюється, його хеш став би іншим; кортеж з незмінних елементів — ні.
5. Щоб різні напрямки одного маршруту потрапили в один лічильник.
6. `1 == 1.0` і `hash(1) == hash(1.0)`: словник вважає їх тим самим ключем.
7. Множина зберігає для кожного елемента ще й hash, тримає щонайменше третину комірок порожніми, а кількість комірок — степінь двійки. `list` — лише посилання по 8 байт.
8. `PRIMARY KEY` — ключ словника; `GROUP BY` — `Counter` (`HASH_GROUP_BY`); `JOIN` за ключем — словник-індекс (`HASH_JOIN`).

</details>

### Шпаргалка

```python
seen = {}                                        # Two Sum: значення → позиція
if target - value in seen: ...                   # O(1) в середньому
by_id = {trip.id: trip for trip in trips}        # індекс: O(n) один раз, далі O(1)
key = tuple(sorted((a, b)))                      # канонічний ключ
Counter(items).most_common(3)                    # частоти
{k: v for k, v in d.items() if v >= 2}           # не видаляй під час обходу — будуй новий
# ключі dict і елементи set — лише hashable: str, int, tuple; list — TypeError
hash(key) % n                                    # номер комірки; 1, 1.0, True — один ключ
sys.getsizeof(obj)                               # лише сам контейнер; set ≈ 5× list
# SQL: PRIMARY KEY ≈ ключ dict, GROUP BY ≈ Counter, JOIN ≈ словник-індекс
```

## Далі

**Урок 18 — огляд модуля 1**: усе разом — типи, колекції, функції, винятки, файли, Git і три практикуми.